In [ ]:
#EXECUTAR ESSE NOTEBOOK NO COLAB PARA PODER USAR A GPU DA GOOGLE

In [1]:
#import numpy as np #eu devo fazer as operações com os tensores (embeddings) usando torch
import numpy as np
import torch
import torch.nn.functional as F
import random
import time

from transformers import BertTokenizer # Or AutoTokenizer
from transformers import BertForPreTraining  # BertForPreTraining for loading pretraining heads; or AutoModelForPreTraining
from transformers import BertModel  # BertModel for BERT without pretraining heads, or AutoModel


model = BertModel.from_pretrained('neuralmind/bert-base-portuguese-cased')
model.to("cuda")
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Funções auxiliares

In [2]:
def tokens_embeddings(frase, tokenizer, model):
  '''retorna os tokens e os embeddings correspondentes a uma frase.
  Usa GPU'''

  inputs = tokenizer(frase, return_tensors='pt') #pt é de pytorch
  inputs.to("cuda")
  with torch.no_grad():
    outputs = model(**inputs)



  # Vetores de embeddings
  embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
  tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

  #tokens_limpo1 = tokens[1:len(embeddings)-1]
  #^pra retornar sem [CLS] e [SEP]

  #Aqui seria a parte de tirar as stopwords da lista de tokens

  #return tokens_limpo1


  return (tokens, embeddings)


def similaridade(emb1, emb2):
  '''retorna a similaridade de cosseno entre dois embeddings
  Funciona por causa do
  import torch.nn.functional as F'''

  return (F.cosine_similarity(emb1, emb2, dim=0)).item() #dentro do parenteses é um tensor


def get_stopwords(stopwords_file):
  '''retorna um conjunto com todas as stopwords presentes
  no arquivo txt stopwords_file'''


  arq_stop = open(stopwords_file, "r")
  set_stopwords = set()


  for line in arq_stop: #o txt tem uma palavra por linha
    word = line.replace("\n", "")

    set_stopwords.add(word)

  arq_stop.close()

  return set_stopwords


def similarity_all_pairs(tensores):
  '''Tentativa de otimizar o cálculo da similaridade entre todos os pares de embeddings'''

  # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = tensores.norm(dim=1, keepdim=True)

  # terminando de normalizar
  data_normalized = tensores / norms

  # Calculando a similaridade de cosseno (all pairs)
  similarity_matrix = torch.matmul(data_normalized, data_normalized.T)

  return similarity_matrix


def normaliza(matriz):
  '''retorna a matriz normalizada para o cálculo da similaridade
  all pairs'''

  # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = matriz.norm(dim=1, keepdim=True) #pegando a norma

  # terminando de normalizar
  data_normalized = matriz / norms


  return data_normalized





def gera_frase(corpus):
  lista_palavras = corpus.split()

  tam = random.randint(2,30)

  frase = ""

  for i in range(tam):
    indice = random.randint(0,len(lista_palavras)-1)
    palavra = lista_palavras[indice]
    frase += palavra

    if(i<(tam-1)):
      frase+=" "


  return frase




## Versão mais otimizada do processa janela

In [3]:

def processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50):
  '''Lê os documentos textuais em cada linha do arquivo de origem, gera os tokens e embeddings e,
  por batches, determina as similaridades de cosseno entre TODOS os pares de embeddings, e determina,
  baseado na similaridade e no limiar (threshold), quais tokens vão se tornar um só na versão processada
  da janela.
  Escreve no arquivo de destino a versão processada da janela, pronta pra gerar redes de cliques
  e calcular as métricas.

  batch_size é a quantidade de documentos textuais analisados num batch.'''

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP] #lista de frases na vdd
                    #cada frase é uma lista com os tokens dessa frase
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  #DEBUG
  k=0
  linha_count = -1

  n_tokens = 0 #número total de tokens

  for line in arq: #aparentemente isso funciona
    linha_count+=1

    frase = line.replace("\n","")
    #frase = line.split(' ')

    #DEBUG
    #if(linha_count<10):
      #print("Linha "+ str(linha_count))

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    #DEBUG
    #if(linha_count < 10):
      #print("\nFrase atual ", tokens)



    tokens_pra_lista = []
    embeddings_pra_lista = []



    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    tam_lista = len(tokens_pra_lista)
    n_tokens += tam_lista

    if(tam_lista>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    k+=1



  arq.close()



  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()

  #Tenho que fazer o primeiro tensorzão aqui
  #Um embedding por linha

  #antes vou fazer a listona_tokens, onde coloco um token apos o outro,
  # pra so no final transformar de novo na lista_tokens pra botar no arquivo
  listona_tokens = []
  for frase in lista_tokens:
    listona_tokens.extend(frase)

  #debug
  #print("\n\n")
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

  lista_tokens = [] #economizando memoria


  tamanho_frases=[] #lista com a quantidade de tokens/embeddings em cada frase, pra facilitar a indexação
  posicao_frase = [] #posição do 1º token de cada frase na matriz (tensorzão) que será construida
  posicao_frase.append(0)  #a primeira frase sempre estará na posição 0
  for i in range(len(lista_embeddings)):
    tamanho_frase = len(lista_embeddings[i])
    tamanho_frases.append(tamanho_frase)

    #aqui colocamos a posição da frase SEGUINTE baseado em onde a atual termina
    posicao_frase.append(posicao_frase[i]+tamanho_frase)

  #posicao_frase vai ficar com uma entrada a mais no final
  posicao_frase = posicao_frase[:len(posicao_frase) - 1]

  #fazendo o tensorzao pra calcular as similaridades

  tam = len(lista_embeddings[0][0])
  print("Tamanho de cada embedding: ", tam)
  matriz = torch.zeros([n_tokens, tam], dtype=torch.float32, device='cuda')
  indice_matriz = 0
  for i in range(len(lista_embeddings)):
    for j in range(len(lista_embeddings[i])):
      matriz[indice_matriz] = lista_embeddings[i][j]
      indice_matriz += 1


  matriz_norm = normaliza(matriz) #agora tenho a matriz completa, já normalizada
  #lembrando, é um tensor
  matriz_norm.to('cuda') #já está?

  #tamanho_frases_torch = torch.tensor(tamanho_frases)
  #tamanho_frases_torch.to('cuda')



  #Agora tenho que popular a lista final de tokens
  #descomentar?
  # tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  #DEBUG
  print("separando a matrizona em matrizinhas\n")

  #dividindo os batches
  n_frases = len(lista_embeddings)

  print("\n\n Total de "+str(n_frases)+" frases(documentos)")
  qtd_batches = int(n_frases/batch_size) #vai arredondar pra baixo
  #essa qtd_batches tbm é a quantidade de rodadas!

  matrizinhas = [] #A_0, A_1, A_2...A_(qtd_batches-1)
  #por enquanto isso é uma lista de tensores... tensores esses que estão em GPU
  #Eu deveria fazer um tensor tri-dimensional?
  #Obs: não parece ser aqui o gargalo

  qtd_tokens_batches = [] #vou usar dps

  #fim_cada_batch = [] #vou usar dps, na hora de comparar similaridades

  fim_batch_ant = 0  #referente aos tokens
  for i in range(qtd_batches):
    inicio_batch_i = fim_batch_ant

    if(i == (qtd_batches-1) ): #se é o último
      fim_batch_i =  len(matriz_norm)

      #calculando qtd_tokens_batch nesse caso
      qtd_tokens_batch = fim_batch_i- inicio_batch_i
      qtd_tokens_batches.append(qtd_tokens_batch)


    else:
      qtd_tokens_batch = 0
      #calcular a qtd de tokens nesse batch
      frase_init = int(n_frases/qtd_batches) * i #primeira frase desse batch

      f_atual = frase_init #isso reflete a posição da frase em tamanho_frases

      for j in range(batch_size): #são batch_size frases em cada matrizinha/ cada batch
        n_tokens_frase = tamanho_frases[f_atual]
        qtd_tokens_batch += n_tokens_frase

        f_atual +=1

      fim_batch_i = inicio_batch_i + qtd_tokens_batch
      qtd_tokens_batches.append(qtd_tokens_batch)

      fim_batch_ant = fim_batch_i #pro próximo loop


    #pegar efetivamente a matrizinha correspondente e botar na lista

    #os indices de slice agr sao inicio_batch_i, fim_batch_i

    matrizinha = matriz_norm[inicio_batch_i: fim_batch_i]

    matrizinhas.append(matrizinha)

  #Obs: agr tenho em memória 2x qtd de tokens x 768 (o tamanho do tensor)
  # x 4 bytes, pe é float32

  #agora falta, em cada batch, fazer os devidos produtos e colocar
  #numa nova matriz, horizontalmente
  #   (usando torch.hstack(lista_de_tensores) )
  #é aqui que fica pesado em termos de memória


  indice_origem = 0 #indice do token "origem" na listona_tokens
                    #Vai aumentar de 1 em 1 a cada vez que olharmos
                    #pra uma lista de similaridades dentro da lista de
                    #de listas de similaridades

  conjuntos = [] #componente conexo ao qual cada token pertence
                        #considerando o grafo formado pela matriz binária
                        #de similaridades


  for i in range(n_tokens): #começa do 0, e tá tudo bem
    conjuntos.append(i)
  #originalmente, cada token pertence ao seu
  #proprio componente conexo

  conjuntos_tokens = torch.tensor(conjuntos, device='cuda')

  conjuntos = [] #jogando fora a lista, ficando so com o tensor
  posicoes_justapostas = set()

  print("Entrando no forzao")


  for b in range(qtd_batches):  #b de batch
    #preciso preparar a matriz de similaridades referente a
    #linha do batch atual na matrizona de similaridades que
    #está no mundo das ideias (vide rascunho)

    #vou fazer os cálculos dos produtos de matrizes menores
    #primeiro.
    #  dps eu boto numa matriz maior que vai ser a referente a essa rodada.

    #descobrindo quantos produtos (A_i * A_j transposta)nessa rodada

    #DEBUG
    if(b==0):
      print('\n\n primeiro batch')
    elif(b==1):
      print('\n\nsegundo batch')
    else:
      print('\n\nbatch '+str(b))


    n_prod = qtd_batches - b #na rodada zero, sera qtd_batches,
                             #dps qtd_batches-1, por ai vai

    #tam_embedding = 768
    qtd_tokens_ate_final = qtd_tokens_batches[b:] #do atual pra frente
    #qtd_tokens_ate_final é uma lista, guarda a qtd de tokens em cada batch até o final
    soma_tokens_batches = sum(qtd_tokens_ate_final) #pior caso: 2M e pouco...



    #pior caso: 70*30 X 70*30* qtd_batches * 8 (bytes)
    # =         70*30 X 70*30 * (5000/70) * 8 =~ 2.5 GB
    #similaridades = matriz = torch.zeros([qtd_tokens_batch, soma_tokens_batches], dtype=torch.float64, device='cuda')
    #vai ser o tensor com as similaridades
    #referentes a cada batch, dispostas horizontalmente (um A*A.T do lado do outro).
    # IMPORTANTE:
    # esse tensor será sobrescrito a cada batch.
    #É assim que economizamos memória!
    #OBS: Com o hstack, nao vou usar....

    similaridades = 1 #esse 1 é dummy, similaridades vai ser a matriz
                      #com as similaridades dessa rodada organizadas horizontalmente
    lista_tensores = [] #vou colocar os produtos aqui pra depois fazer o hstack deles

    #DEBUG
    #print("Multiplicando as matrizes do batch ", b)
    #print()
    for p in range(n_prod): #p de produto

      #vamos efetuar as multiplicações aqui

      matriz1 = matrizinhas[b] #fixa pra esse batch!!
      matriz2 = matrizinhas[b+p] #inicialmente vai ser ela com ela mesma

      produto = torch.matmul(matriz1, matriz2.T)

      lista_tensores.append(produto)

    #dps de fazer todos os produtos
    similaridades = torch.hstack(lista_tensores)

    #DEBUG
    #if(b<2):
      #sample = similaridades[:2]
      #print("\nsimilaridades (original): ")
      #print(sample)
      #sample = []

    lista_tensores = [] #pra economizar memoria

    #############################################################
    # Falta agora fazer as junções a partir
    #das similaridades!


    #MAS ANTES: Conversão em matriz binária, e multiplicar por [1 2 3 ...]
    # e tirar os zeros

    n_linhas = qtd_tokens_batches[b] #qtd de tokens nesse batch
    n_col = soma_tokens_batches #qtd de tokens até o final da matrizona teórica

    #matriz_p_soma = torch.ones(n_linhas, n_col, device = 'cuda') * (1-threshold)
    #ocupa espaço!!

    vetor_p_soma = torch.ones(1, n_col, device = 'cuda') * (1-threshold)


    #print(vetor_p_soma)

    similaridades = similaridades + vetor_p_soma
    #print(similaridades)

    similaridades = similaridades.int()

    #print(T)


    #FAZENDO A SEQUENCIA

    #definir n_col!!! tem que ser igual o numero de tokens!
    sequencia = torch.zeros(1,n_col, device= 'cuda')


    #num = 1
    #for i in range(len(sequencia[0])):
      #sequencia[0][i] = num
      #num +=1
    #esse método desconsidera a separação em batches!

    primeiro_token_batch_atual = 0

    if(b==0): #se é o primeiro batch
      num = 1
      for i in range(n_col):
        sequencia[0][i] = num
        num +=1
      #depois vou tirar um de cada posição nas listas de similaridades

    else: #nao precisamos mais nos preocupar com o token 0
      qtd_tokens_ate_agr = qtd_tokens_batches[:b] #vai retornar uma lista
      primeiro_token_batch_atual = sum(qtd_tokens_ate_agr)
      #posição na listona_tokens do primeiro token do batch atual

      num = primeiro_token_batch_atual
      for i in range(n_col):
        sequencia[0][i] = num
        num +=1


    #DEBUG
    #if(b<2):
      #print("sequencia")
      #print(sequencia)

    similaridades = similaridades*sequencia

    #print(similaridades)
    if(b!=0): #caso normal
      listas =[l [l !=0] for l in similaridades]



    else: #aqui b == 0 e portanto precisamos nos preocupar com o token 0
      listas =[(l [l !=0])-1 for l in similaridades]
      #obs: aqui estamos quase dobrando a memória!
      #obs2: as listas tem tamanhos diferentes
      #obs3: com o -1 no caso do primeiro batch os índices ficam corretos
      #      e capturam corretamente as similaridades pq o -1 foi feito
      #      dps de tirar os zeros!




    similaridades = [] #pra desocupar memória

    #DEBUG
    #if(b<2):
      #print("\n\n primeiras listas de similaridades:")
      #print(listas[0][:30])
      #print(listas[1][:30])


    #"pré-processamento" feito! agora tem que, a partir das
    #listas, redesignar os tokens

    frase_inicial = b*batch_size
    frase_origem = frase_inicial #frase do token "origem": estou olhando pra
                                 #lista de similaridades correspondente a esse token

    #frase_final = (b+1)*batch_size

    #A palavra atual vai ser um lista_tokens[i][j] da vida
    #Temos tamanho_frases, [] uma lista com a qtd de tokens/embeddings em cada
    #  frase;
    #  e posicao_frase, uma lista com a posição de cada frase na matriz
    # (tensorzão) construida com os embeddings um embaixo do outro
    #De posse disso e das infos de batch, e com uma lista pra cada TOKEN do tipo
    #  [1, 45, 789, 2335, 6601, 7009 ...]
    # temos que localizar as palavras origem e destino na listona_tokens pra fazer
    # as alterações necessárias
    #Com a lista_tokens pronta dps do ultimo batch, é só repetir o que é
    # feito nas outras versões dessa função!
    #------OBS:------ pra facilitar, construí uma listona de tokens,
    # "jogando fora" a informação da frase
    # E depois vou colocar os tokens dessa listona na lista_tokens original
    #usando o tamanho_frases
    #OBS: joguei isso pra antes do for, pra fazer uma vez só!
    #listona_tokens = []
    #for frase in lista_tokens:
    #  for token in frase:
    #    listona_tokens.append(token)

    #debug
    #print("\n\n")
    #print(lista_tokens[:2])
    #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

    #DEBUG
    #print("\nVerificando similaridades e redesignando tokens...")

    #0-INDEXADO!!
    #posicoes_justapostas = set() #posições na listona_tokens
                                 #dos tokens que sofreram justaposição

    #indice_origem = 0 #indice do token origem na listona_tokens
    #DEBUG                  #cuidado com o batch! resolvi acima? conferir
    #cont_debug = -1 #pra quando eu acessar pela primeira vez ser 0

    for lista_sim in listas:
      #DEBUG
      #cont_debug += 1

      if(len(lista_sim) < 2): #só é similar a ele próprio
        indice_origem +=1
        continue


      #cada lista representa um token do batch
      #aqui eu posso se quiser fazer um pequeno calculo de qual a frase
      #do token pra que a alteração do token carregue a informação da
      #frase & do token

      #a principio vai ser so o indice da listona_tokens msm

      token_origem = listona_tokens[indice_origem]

      #novo_nome_origem = token_origem #como se ja tivesse sofrido justaposição

      #if(indice_origem not in posicoes_justapostas):
        #novo_nome_origem = token_origem+"_"+str(indice_origem) #to sobrescrevendo
                                                               #a var nome_origem


      #alterou = 0
      #começar a olhar só a partir de indice_origem na lista_sim?
      conjuntos_atuais = torch.tensor([0 for i in range(len(lista_sim)) ], device='cuda')
      #o conjunto (ou componente) de cada token na lista de similaridades atual
      idx_lista_sim = 0
      for num in lista_sim: #indice do token similar ao "token origem"

        inteiro = int(num)

        conjuntos_atuais[idx_lista_sim] = conjuntos_tokens[inteiro]
        idx_lista_sim+=1

      menor = torch.min(conjuntos_atuais, dim=0)
      #vai ter "values" e "indices"
      menor_valor = int(menor.values)

      #DEBUG
      #if((b<2) and (cont_debug<2)):
        #print("\nmenor componente conexo na lista: ", menor_valor)

      indice_menor = int(menor.values)
      #indice_menor =int(menor.indices) +primeiro_token_batch_atual #errado?
      #indice na listona_tokens do token que esta associado ao componente conexo
      #de menor numero
      #OBS: O token de menor indice associado ao componente conexo de menor
      # numero (seja x esse numero) sempre estará na posição x na listona_tokens
      #por construção: todo mundo começa associado ao componente de seu próprio indice
      #e todo mundo que vai pro mesmo componente, vai pro componente de menor número

      if(indice_menor not in posicoes_justapostas):
        nome_final = listona_tokens[indice_menor]+"_"+str(menor_valor)
        posicoes_justapostas.add(indice_menor)

      else: #pra nao ficar fazendo "palavra_indice_indice_indice_indice..."
            #a cada justaposição
        nome_final = listona_tokens[indice_menor]
            #ja tem o "_indice" no final

      #DEBUG
      #if((b<2) and (cont_debug<2)):
        #print("\n Nome final dos tokens similares: ", nome_final)
      #É IMPORTANTÍSSIMO que isso aqui esteja certo pra que dê certo globalmente!!
      #Assim, todos ficam com o numero do componente conexo correspondente.
      #Que por sua vez tem esse numero, originalmente, por causa do primeiro token associado
      #a ele. Efetivamente, todos os termos semelhantes entre si ficam com o
      #indice na listona_tokens do primeiro termo que é semelhante a todos eles
      #("semelhante" aqui é quem está no mesmo componente conexo considerando
      #o grafão teórico das similaridades completas)


      for num in lista_sim:

        inteiro = int(num)

        #passando geral pro menor conjunto (o conjunto de menor numero)
        conjuntos_tokens[inteiro] = menor_valor
        #renomeando
        listona_tokens[inteiro] = nome_final
        #posicoes_justapostas.add(inteiro) #nao precisa...
        #note que, por causa da forma que a sequencia foi montada,
        #os índices já estão certos,
        #independente do batch


        #indice_destino = num


        #aqui, se o token destino já tiver sido justaposto,
        #O token_origem é que tem que mudar de nome ?????
        #listona_tokens[indice_destino] = novo_nome_origem
        #posicoes_justapostas.add(indice_destino)
        #alterou = 1

        #^^^ todo mundo que estiver na lista vai ter seu nome alterado
        #    pro nome do token original + "_posicaoOriginal"


      #saiu do for
      #if(alterou != 0): #ou seja, se rolou alguma justaposição
        #listona_tokens[indice_origem] = novo_nome_origem
        #posicoes_justapostas.add(indice_origem)



      indice_origem += 1


    #print("\n\nlistona_tokens pronta")
    #print("reconstruindo a lista_tokens")




    #ir pro próximo batch!
    #dps só falta reconstruir a janela assim como nos outros

  #Saí do forzão
  #reconstruindo a lista_tokens

  #####################01/04:verifiquei CORRETUDE até aqui, falta limpar!#######
  #DESCOMENTAR
  print("\n\n Saímos do forzão! similaridades já avaliadas\n Reconstruindo lista_tokens...")

  #DEBUG
  #time.sleep(10)

  listas = [] #lista de listas de similaridades
              #liberando a memória pois não vamos mais usar

  idx_listona = 0
  for tamanho in tamanho_frases:
    idx_frase = 0
    tokens_pra_lista = []
    while(idx_frase < tamanho):
      tokens_pra_lista.append(listona_tokens[idx_listona])
      idx_listona += 1
      idx_frase +=1

    lista_tokens.append(tokens_pra_lista) #uma frase


  #debug
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda


  print("\nescrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w+') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  return





def processa_varias_janelas(arquivos_sem_ext, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50):
  '''Processa varias janelas (os nomes dos arquivos origem estão na lista "arquivos_sem_ext", só
  com o nome, sem a extensão .txt) com o processa_janela_bert_fast; os arquivos resultantes são salvos em
  "nome_arquivo_Processada.txt"  '''

  for nome_arq in arquivos_sem_ext:
    origem = nome_arq+".txt"
    destino = nome_arq+"_Processada.txt"

    processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold, batch_size)
    torch.cuda.empty_cache()


In [ ]:
#Testando com uma janela real!! quase 10.000 tweets

nome_arq = "T62.txt"
destino = "T62_Processada.txt"
origem = nome_arq
stopwords_file = "stopwords_pt.txt"

processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=80)
torch.cuda.empty_cache()

lendo arquivo T62.txt

lista de tokens e embeddings pronta. Restaram 144383 tokens e 9631 frases

Tamanho de cada embedding:  768
separando a matrizona em matrizinhas



 Total de 9631 frases(documentos)
Entrando no forzao


 primeiro batch


segundo batch


batch 2


batch 3


batch 4


batch 5


batch 6


batch 7


batch 8


batch 9


batch 10


batch 11


batch 12


batch 13


batch 14


batch 15


batch 16


batch 17


batch 18


batch 19


batch 20


batch 21


batch 22


batch 23


batch 24


batch 25


batch 26


batch 27


batch 28


batch 29


batch 30


batch 31


batch 32


batch 33


batch 34


batch 35


batch 36


batch 37


batch 38


batch 39


batch 40


batch 41


batch 42


batch 43


batch 44


batch 45


batch 46


batch 47


batch 48


batch 49


batch 50


batch 51


batch 52


batch 53


batch 54


batch 55


batch 56


batch 57


batch 58


batch 59


batch 60


batch 61


batch 62


batch 63


batch 64


batch 65


batch 66


batch 67


batch 68


batch 69


b

In [ ]:
#Processando varias janelas
#Lembrar de uploadar os arquivos correspondentes e o de stopwords
stopwords_file = "stopwords_pt.txt"
arquivos_sem_ext = ["T63", "T64"]


processa_varias_janelas(arquivos_sem_ext, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50)